# Assignment 4: Feature Extraction, Face Recognition, and Parameter Detection

**Student**: Marston Ward  
**Course**: AAI-521 Computer Vision  

This notebook is compatible with both **M1/M2/M3 Silicon Mac** and **Google Colab**.

## Setup and Imports

In [ ]:
# Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
from pathlib import Path

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab.patches import cv2_imshow
    print("Running in Google Colab")
else:
    print("Running locally")

# Setup device for PyTorch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
# Helper function for displaying images (cross-platform)
def show_image(img, title="Image", cmap=None):
    """Display image - works for both Colab and local environments"""
    if IN_COLAB:
        print(f"\n{title}:")
        cv2_imshow(img)
    else:
        plt.figure(figsize=(10, 8))
        if len(img.shape) == 3 and img.shape[2] == 3:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(img, cmap=cmap if cmap else 'gray')
        plt.title(title)
        plt.axis('off')
        plt.show()

def show_images_side_by_side(img1, img2, title1="Image 1", title2="Image 2"):
    """Display two images side by side"""
    if IN_COLAB:
        print(f"\n{title1}:")
        cv2_imshow(img1)
        print(f"\n{title2}:")
        cv2_imshow(img2)
    else:
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        
        if len(img1.shape) == 3:
            axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
        else:
            axes[0].imshow(img1, cmap='gray')
        axes[0].set_title(title1)
        axes[0].axis('off')
        
        if len(img2.shape) == 3:
            axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
        else:
            axes[1].imshow(img2, cmap='gray')
        axes[1].set_title(title2)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()

print("Helper functions loaded successfully!")

---
## Part 1: Feature Extraction

In this section, deals with different feature detection algorithms:
- **SIFT** (Scale-Invariant Feature Transform)
- **FAST** (Features from Accelerated Segment Test)
- **ORB** (Oriented FAST and Rotated BRIEF)

I'll also perform feature matching between color and grayscale versions of the image.

### Load and Prepare Images

In [ ]:
# Read the image (it loads as BGR by default)
img_bgr = cv2.imread('Assignment4_pic1.jpg')

if img_bgr is None:
    raise FileNotFoundError("Assignment4_pic1.jpg not found. Please ensure the file is in the working directory.")

# Convert BGR to RGB for proper color display
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Convert to grayscale
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Save grayscale image as required
cv2.imwrite('pic1_gray.jpg', img_gray)
print("Saved pic1_gray.jpg")

# Display both versions
print(f"Image shape (RGB): {img_rgb.shape}")
print(f"Image shape (Gray): {img_gray.shape}")

show_images_side_by_side(img_bgr, img_gray, "Original Image (RGB)", "Grayscale Image")

### Part 1a: SIFT (Scale-Invariant Feature Transform)

**SIFT** was introduced by David Lowe in 2004. It detects distinctive features that are invariant to scale, rotation, and illumination changes.

In [ ]:
# Initialize SIFT detector
sift = cv2.SIFT_create()

# Detect keypoints and compute descriptors
keypoints_sift, descriptors_sift = sift.detectAndCompute(img_gray, None)

print(f"Number of SIFT keypoints detected: {len(keypoints_sift)}")

# Draw keypoints WITHOUT size information
img_sift_no_size = cv2.drawKeypoints(img_bgr, keypoints_sift, None, 
                                      flags=cv2.DRAW_MATCHES_FLAGS_DEFAULT)

# Draw keypoints WITH size information
img_sift_with_size = cv2.drawKeypoints(img_bgr, keypoints_sift, None, 
                                        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

show_images_side_by_side(img_sift_no_size, img_sift_with_size, 
                        "SIFT without Size", "SIFT with Size & Orientation")

#### Analysis: SIFT Features

SIFT detected distinctive keypoints showing circles of different sizes (scale) and orientation lines. SIFT is robust to scale changes, rotation, illumination, and partial occlusion, making it good for feature matching. However, it can be computationally expensive. Just looking that image with the invariant features doesn't lend itself to easy interpretation. What is seems to be showing are features that remain if you transform the image.

### Part 1b: FAST (Features from Accelerated Segment Test)

**FAST** (2006) is a corner detection method designed for real-time video processing.

In [ ]:
# Initialize FAST detector WITHOUT non-maximum suppression (NMS)
# NMS is a technique used to select one entity among many overlapping entities
fast_no_nms = cv2.FastFeatureDetector_create()
fast_no_nms.setNonmaxSuppression(False)
keypoints_fast_no_nms = fast_no_nms.detect(img_gray, None)
print(f"FAST keypoints without NMS: {len(keypoints_fast_no_nms)}")

# Initialize FAST detector WITH non-maximum suppression (NMS)
fast_with_nms = cv2.FastFeatureDetector_create()
fast_with_nms.setNonmaxSuppression(True)
keypoints_fast_with_nms = fast_with_nms.detect(img_gray, None)
print(f"FAST keypoints with NMS: {len(keypoints_fast_with_nms)}")

# Draw keypoints
img_fast_no_nms = cv2.drawKeypoints(img_bgr, keypoints_fast_no_nms, None, color=(0, 255, 0))
img_fast_with_nms = cv2.drawKeypoints(img_bgr, keypoints_fast_with_nms, None, color=(0, 255, 0))

show_images_side_by_side(img_fast_no_nms, img_fast_with_nms, "FAST without NMS", "FAST with Non-Max Suppression")

#### Analysis: FAST Features

FAST detects corners quickly by examining a circle of 16 pixels. Non-maximum suppression removes redundant adjacent keypoints. Without NMS shows many clustered points, while with NMS shows better-distributed keypoints. FAST is ideal for real-time applications but doesn't provide scale or rotation invariance. Notice how the eyes and edges of the hair are fairly well outlined. This is also true for the eyebrows. But mouth was surprisingly not well outlined as I expected - although it is better with NMS. 

### Part 1c: ORB (Oriented FAST and Rotated BRIEF)

**ORB** (2011) combines FAST keypoint detection with BRIEF descriptors, providing a fast, free alternative to SIFT.

In [ ]:
# Initialize ORB detector
orb = cv2.ORB_create(nfeatures=1500)

# Detect keypoints and compute descriptors
keypoints_orb, descriptors_orb = orb.detectAndCompute(img_gray, None)

print(f"Number of ORB keypoints detected: {len(keypoints_orb)}")

# Draw keypoints with orientation
img_orb = cv2.drawKeypoints(img_bgr, keypoints_orb, None, 
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
                            color=(255, 0, 0))

show_image(img_orb, "ORB Features with Orientation")

#### Analysis: ORB Features

ORB provides excellent balance: ~100x faster than SIFT, rotation invariant, scale aware, and uses binary descriptors for fast matching. Ideal for real-time AR, SLAM, and object tracking. I really like this combination of techniques as it is far superior to SIFT and FAST individually. 

### Part 1d: Feature Matching Between Color and Grayscale Images

In [ ]:
# Detect features in both images
keypoints_gray, descriptors_gray = orb.detectAndCompute(img_gray, None)
keypoints_color, descriptors_color = orb.detectAndCompute(img_bgr, None)

print(f"Keypoints in grayscale: {len(keypoints_gray)}")
print(f"Keypoints in color: {len(keypoints_color)}")

# Create BFMatcher and match
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(descriptors_gray, descriptors_color)
matches = sorted(matches, key=lambda x: x.distance)

print(f"\n✓ Number of matching points: {len(matches)}")
print(f"Average match distance: {np.mean([m.distance for m in matches]):.2f}")

# Draw top 100 matches
img_matches = cv2.drawMatches(img_gray, keypoints_gray, img_bgr, keypoints_color, 
                              matches[:100], None, 
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

show_image(img_matches, "Top 100 Feature Matches: Grayscale vs Color")

#### Analysis: Feature Matching

The high match count and low distances confirm feature descriptors are color-invariant. This enables object recognition, image stitching, 3D reconstruction, and motion tracking across different image representations. I was really surprised by how well the lines matched up. This has given me some ideas about using this tool to detect certain weather patterns. 

---
## Part 2: Face Recognition (Face and Eye Detection)

Using **Haar Cascade Classifiers** for face and eye detection.

In [ ]:
# Load Haar Cascade classifiers
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

if face_cascade.empty() or eye_cascade.empty():
    raise FileNotFoundError("Haar cascade XML files not found")

print("✓ Haar Cascade classifiers loaded successfully")

### Part 2a: Face and Eye Detection on Static Image

In [ ]:
# Read image
img_faces = cv2.imread('Assignment4_pic2.jpg')
if img_faces is None:
    raise FileNotFoundError("Assignment4_pic2.jpg not found")

gray_faces = cv2.cvtColor(img_faces, cv2.COLOR_BGR2GRAY)

# Detect faces
faces = face_cascade.detectMultiScale(gray_faces, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
print(f"Number of faces detected: {len(faces)}")

img_detected = img_faces.copy()
total_eyes = 0

# the faces variable have (x, y, w, h) for each detected face
for i, (x, y, w, h) in enumerate(faces):
    cv2.rectangle(img_detected, (x, y), (x+w, y+h), (255, 0, 0), 3)
    cv2.putText(img_detected, f'Face {i+1}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    
    # Detect eyes in face region
    roi_gray = gray_faces[y:y+h, x:x+w]
    roi_color = img_detected[y:y+h, x:x+w]
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=5, minSize=(20, 20))
    total_eyes += len(eyes)
    
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

print(f"Number of eyes detected: {total_eyes}")

show_images_side_by_side(img_faces, img_detected, "Original Image", 
                        f"Detected: {len(faces)} Faces, {total_eyes} Eyes")

#### Analysis: Face Detection

Haar cascades use rectangular features to detect light/dark patterns. They work best with frontal faces and good lighting. Limitations include poor performance on profile views, sensitivity to occlusions and lighting. Modern deep learning alternatives (MTCNN, RetinaFace) offer better accuracy. As you can see, while this image was as good use case, the model still had some false positives.

### Part 2b: Real-Time Webcam Detection

In [15]:
if not IN_COLAB:
    import time
    
    print("Starting webcam detection...")
    print("Press 'q' to quit\n")
    
    # Try to access webcam
    cap = cv2.VideoCapture(0)
    
    # Check if camera opened successfully
    if not cap.isOpened():
        print("\n⚠️  ERROR: Could not access webcam")
        print("\nTroubleshooting steps for macOS:")
        print("1. Open System Settings → Privacy & Security → Camera")
        print("2. Grant camera permission to Terminal/Python/Your IDE")
        print("3. Close and reopen your terminal/IDE after granting permission")
        print("4. Run this cell again")
        print("\nAlternative: Use 'Part 2b Alternative' cell below\n")
    else:
        print("✓ Webcam opened successfully!")
        
        # IMPORTANT: M4 Mac needs longer initialization time
        # Try to read a frame multiple times to ensure camera is ready
        print("⏳ Initializing camera (M4 Mac may take a moment)...")
        
        camera_ready = False
        for attempt in range(10):
            time.sleep(0.3)
            ret, test_frame = cap.read()
            if ret and test_frame is not None:
                camera_ready = True
                print(f"✓ Camera ready after {(attempt+1)*0.3:.1f} seconds\n")
                break
        
        if not camera_ready:
            print("\n❌ ERROR: Camera opened but could not capture frames")
            print("\nPossible causes:")
            print("1. Camera is in use by another application")
            print("2. Hardware/driver issue")
            print("3. Need to restart computer\n")
            print("Solution: Try the 'Part 2b Alternative' cell with a static photo\n")
            cap.release()
        else:
            # Camera is working, start detection loop
            frame_count = 0
            face_detection_count = 0
            
            try:
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        print("\nLost camera connection")
                        break
                    
                    frame_count += 1
                    
                    # Convert to grayscale and detect faces
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
                    
                    if len(faces) > 0:
                        face_detection_count += 1
                    
                    # Draw rectangles around faces and detect eyes
                    for (x, y, w, h) in faces:
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
                        
                        roi_gray = gray[y:y+h, x:x+w]
                        roi_color = frame[y:y+h, x:x+w]
                        eyes = eye_cascade.detectMultiScale(roi_gray)
                        
                        for (ex, ey, ew, eh) in eyes:
                            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)
                    
                    # Display info on frame
                    cv2.putText(frame, f'Faces: {len(faces)}', (10, 30), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    cv2.putText(frame, "Press 'q' to quit", (10, 70), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                    # Show frame
                    cv2.imshow('Face Detection - Press q to quit', frame)
                    
                    # Break on 'q' key
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
            
            except KeyboardInterrupt:
                print("\nInterrupted by user")
            
            finally:
                # Release camera and close windows
                cap.release()
                cv2.destroyAllWindows()
                
                if frame_count > 0:
                    print(f"\n{'='*50}")
                    print(f"WEBCAM SESSION COMPLETED")
                    print(f"{'='*50}")
                    print(f"Total frames processed: {frame_count}")
                    print(f"Frames with face detected: {face_detection_count}")
                    print(f"Detection rate: {face_detection_count/frame_count*100:.1f}%")
                    print(f"{'='*50}\n")
else:
    print("Running in Google Colab - webcam requires JavaScript capture")
    print("See alternative cell for Colab implementation")

Starting webcam detection...
Press 'q' to quit

✓ Webcam opened successfully!
✓ Camera initialized

Error: Could not read frame


---
## Part 3: Apple Counting

Counting apples using color segmentation and contour detection.

In [ ]:
# Read apple image
img_apples = cv2.imread('apple.jpg')
if img_apples is None:
    raise FileNotFoundError("apple.jpg not found")

show_image(img_apples, "Original Apple Image")

In [ ]:
# Convert to HSV for color segmentation
hsv = cv2.cvtColor(img_apples, cv2.COLOR_BGR2HSV)

# Define red color ranges (red wraps around in HSV)
lower_red1 = np.array([0, 50, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 50, 50])
upper_red2 = np.array([180, 255, 255])

# Create and combine masks
mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
mask = cv2.bitwise_or(mask1, mask2)

# Apply morphological operations
kernel = np.ones((5, 5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
mask = cv2.GaussianBlur(mask, (5, 5), 0)

show_image(mask, "Binary Mask (Red Apples)", cmap='gray')

In [ ]:
# Find and filter contours
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
min_area = 500
apple_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]

print(f"Contours after filtering: {len(apple_contours)}")

# Draw results
img_result = img_apples.copy()
cv2.drawContours(img_result, apple_contours, -1, (0, 255, 0), 3)

for i, contour in enumerate(apple_contours):
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        cv2.circle(img_result, (cX, cY), 7, (255, 0, 0), -1)
        cv2.putText(img_result, str(i+1), (cX - 20, cY - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 0), 3)

cv2.putText(img_result, f'Total Apples: {len(apple_contours)}', (10, 50),
           cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)

print(f"\n{'='*50}")
print(f"FINAL APPLE COUNT: {len(apple_contours)}")
print(f"{'='*50}\n")

show_image(img_result, f"Detected {len(apple_contours)} Apples")

#### Analysis: Apple Counting

Used HSV color space for red segmentation, applied morphological operations to clean the mask, and detected contours filtered by area. This method works well for colored objects against contrasting backgrounds but is sensitive to lighting. Applications include agricultural monitoring, quality control, and inventory management.

---
## Conclusion

This assignment covered feature extraction (SIFT, FAST, ORB), face detection with Haar cascades, and object counting with color segmentation. These techniques form the foundation for advanced computer vision applications including tracking, 3D reconstruction, and visual recognition systems.